# Data Vault Walkthrough

This notebook demonstrates the creation and querying of the Data Vault for the 'Track 1' recovered data.

## 1. Setup and Ingestion

First, we initialize the `DataVaultBuilder` and run the ingestion process. This will process:
- Master CSVs (Wells, Surveys, Sensors)
- Archive Batch Parquet files (Seismic Readings)
- Legacy Survey Parquet files

In [1]:
import pandas as pd
import os
from data_vault_builder import DataVaultBuilder

# Initialize and run ingestion
# Note: If data_vault directory exists, it will append/update. 
# For a clean run, you might want to clear the directory first.
builder = DataVaultBuilder()
builder.run_all()

Data Vault Builder initialized. Output directory: data_vault
Ingesting Masters...
Saved hub_well with 20 rows (or appended/updated).
Saved sat_well_details with 20 rows (or appended/updated).
Saved hub_survey with 3 rows (or appended/updated).
Saved sat_survey_details with 3 rows (or appended/updated).
Saved hub_sensor with 50 rows (or appended/updated).
Saved sat_sensor_details with 50 rows (or appended/updated).
Ingesting Archive Batches...
Skipping corrupted file: .\track_1_forensics\archive_batch_seismic_readings.parquet
Processing archive_batch_seismic_readings_2.parquet with 4570 rows.
Saved hub_well with 13 rows (or appended/updated).
Saved hub_survey with 3 rows (or appended/updated).
Saved hub_sensor with 20 rows (or appended/updated).
Saved link_seismic_trace with 32 rows (or appended/updated).
Saved sat_seismic_reading with 4570 rows (or appended/updated).
Ingesting Legacy Surveys...
Processing legacy_survey_1991_101.parquet (Year: 1991) with 166 rows.
Saved hub_well with 1 

## 2. Inspecting the Data Vault

Now that the data is ingested, let's look at the created tables (Parquet files).

In [2]:
output_dir = 'data_vault'
files = [f for f in os.listdir(output_dir) if f.endswith('.parquet')]
print("Created Tables:")
for f in sorted(files):
    print(f" - {f}")

Created Tables:
 - hub_sensor.parquet
 - hub_survey.parquet
 - hub_well.parquet
 - link_legacy_trace.parquet
 - link_seismic_trace.parquet
 - sat_legacy_reading.parquet
 - sat_seismic_reading.parquet
 - sat_sensor_details.parquet
 - sat_survey_details.parquet
 - sat_well_details.parquet


### 2.1 Hubs
Hubs contain the business keys and unique identifiers.

In [3]:
hub_well = pd.read_parquet(os.path.join(output_dir, 'hub_well.parquet'))
print(f"Hub Well: {len(hub_well)} rows")
hub_well.head()

Hub Well: 20 rows


,well_hash_key,well_id,load_date,record_source
0,c4ca4238a0b923820dcc509a6f75849b,1,2025-12-14 00:16:00.973221,master_wells.csv
1,c81e728d9d4c2f636f067f89cc14862c,2,2025-12-14 00:16:00.973221,master_wells.csv
2,eccbc87e4b5ce2fe28308fd9f2a7baf3,3,2025-12-14 00:16:00.973221,master_wells.csv
3,a87ff679a2f3e71d9181a67b7542122c,4,2025-12-14 00:16:00.973221,master_wells.csv
4,e4da3b7fbbce2345d7772b0674a318d5,5,2025-12-14 00:16:00.973221,master_wells.csv


### 2.2 Links
Links associate Hubs together.

In [4]:
link_seismic = pd.read_parquet(os.path.join(output_dir, 'link_seismic_trace.parquet'))
print(f"Link Seismic Trace: {len(link_seismic)} rows")
link_seismic.head()

Link Seismic Trace: 32 rows


,link_hash_key,well_hash_key,survey_hash_key,sensor_hash_key,load_date,record_source
0,0b4157b3f46a026bc85de00dca619c4a,9bf31c7ff062936a96d3c8bd1f8f2ff3,ec8956637a99787bd197eacd77acce5e,5f93f983524def3dca464469d2cf9f3e,2025-12-14 00:16:00.973221,archive_batch_seismic_readings_2.parquet
1,96e84a802de1c39af7159139aeeccdcb,c4ca4238a0b923820dcc509a6f75849b,38b3eff8baf56627478ec76a704e9b52,9b8619251a19057cff70779273e95aa6,2025-12-14 00:16:00.973221,archive_batch_seismic_readings_2.parquet
2,da72077af1c9bb3b94acc58a8354bf60,c20ad4d76fe97759aa27a0c99bff6710,ec8956637a99787bd197eacd77acce5e,ec5decca5ed3d6b8079e2e7e7bacc9f2,2025-12-14 00:16:00.973221,archive_batch_seismic_readings_2.parquet
3,9f86400e6187379139dd422210dec769,1f0e3dad99908345f7439f8ffabdffc4,ec8956637a99787bd197eacd77acce5e,069059b7ef840f0c74a814ec9237b6ec,2025-12-14 00:16:00.973221,archive_batch_seismic_readings_2.parquet
4,d358087d0acef3714abaf7877dbee496,9bf31c7ff062936a96d3c8bd1f8f2ff3,eae27d77ca20db309e056e3d2dcd7d69,5f93f983524def3dca464469d2cf9f3e,2025-12-14 00:16:00.973221,archive_batch_seismic_readings_2.parquet


### 2.3 Satellites
Satellites contain the historical data and attributes.

In [5]:
sat_reading = pd.read_parquet(os.path.join(output_dir, 'sat_seismic_reading.parquet'))
print(f"Sat Seismic Reading: {len(sat_reading)} rows")
sat_reading.head()

Sat Seismic Reading: 4570 rows


,link_hash_key,depth_ft,amplitude,reading_timestamp,quality_flag,load_date,record_source,hash_diff,file_checksum
0,0b4157b3f46a026bc85de00dca619c4a,3030.303030,0.021171,1998-06-20,0,2025-12-14 00:16:00.973221,archive_batch_seismic_readings_2.parquet,ffbed911cbee47179c10d2b84bbe7c34,5e588a73c27296a46de711c9381ab42d
1,96e84a802de1c39af7159139aeeccdcb,1393.939394,0.358876,1995-04-09,1,2025-12-14 00:16:00.973221,archive_batch_seismic_readings_2.parquet,9fb4c7e33174099ea1c6b2f940e9fd5c,5e588a73c27296a46de711c9381ab42d
2,da72077af1c9bb3b94acc58a8354bf60,421.686747,-4.846975,1999-09-29,1,2025-12-14 00:16:00.973221,archive_batch_seismic_readings_2.parquet,00b1228aa5b6991e27af4ae581432071,5e588a73c27296a46de711c9381ab42d
3,9f86400e6187379139dd422210dec769,818.181818,3.660682,1995-09-27,1,2025-12-14 00:16:00.973221,archive_batch_seismic_readings_2.parquet,774bac4e97b87f46a9bdc0c0562fd12d,5e588a73c27296a46de711c9381ab42d
4,d358087d0acef3714abaf7877dbee496,939.393939,-4.221923,1997-09-16,1,2025-12-14 00:16:00.973221,archive_batch_seismic_readings_2.parquet,f66a076e14dc47742de0fcc164874f08,5e588a73c27296a46de711c9381ab42d


## 3. Reconstructing Data

We can join Hubs, Links, and Satellites to reconstruct the original view or create analytics tables.
Example: Get seismic readings with Well Name and Survey Type.

In [6]:
# Load needed tables
hub_well = pd.read_parquet(os.path.join(output_dir, 'hub_well.parquet'))
hub_survey = pd.read_parquet(os.path.join(output_dir, 'hub_survey.parquet'))
sat_well = pd.read_parquet(os.path.join(output_dir, 'sat_well_details.parquet'))
sat_survey = pd.read_parquet(os.path.join(output_dir, 'sat_survey_details.parquet'))
link_seismic = pd.read_parquet(os.path.join(output_dir, 'link_seismic_trace.parquet'))
sat_reading = pd.read_parquet(os.path.join(output_dir, 'sat_seismic_reading.parquet'))

# Join Link with Sat Reading
df_joined = sat_reading.merge(link_seismic, on='link_hash_key')

# Join with Hubs (to get keys) and then Sats (to get names)
# Note: In a real DV query, we would join on Hash Keys directly

# Get Well Name
df_joined = df_joined.merge(sat_well[['well_hash_key', 'well_name']], on='well_hash_key', how='left')

# Get Survey Type
df_joined = df_joined.merge(sat_survey[['survey_hash_key', 'survey_type']], on='survey_hash_key', how='left')

# Select relevant columns
result = df_joined[['reading_timestamp', 'well_name', 'survey_type', 'depth_ft', 'amplitude', 'quality_flag']]

print("Reconstructed Data (First 10 rows):")
result.head(10)

Reconstructed Data (First 10 rows):


,reading_timestamp,well_name,survey_type,depth_ft,amplitude,quality_flag
0,1998-06-20,CASPIAN-015,Acoustic_3D,3030.303030,0.021171,0
1,1995-04-09,CASPIAN-001,Acoustic_2D,1393.939394,0.358876,1
2,1999-09-29,CASPIAN-012,Acoustic_3D,421.686747,-4.846975,1
3,1995-09-27,CASPIAN-019,Acoustic_3D,818.181818,3.660682,1
4,1997-09-16,CASPIAN-015,Acoustic_VS,939.393939,-4.221923,1
5,1998-04-08,CASPIAN-013,Acoustic_2D,3969.696970,-10.434326,1
6,1996-09-27,CASPIAN-011,Acoustic_3D,3855.421687,-5.022649,1
7,1995-09-27,CASPIAN-019,Acoustic_3D,393.939394,-7.076081,1
8,1995-07-17,CASPIAN-019,Acoustic_VS,2727.272727,7.911632,1
9,1998-06-20,CASPIAN-015,Acoustic_3D,2575.757576,-11.150306,1


## 4. Verification Tests

Running the automated tests to verify data integrity.

In [7]:
!python data_vault_tests.py

.......
----------------------------------------------------------------------
Ran 7 tests in 0.172s

OK
